FEATURE ENGINEERING:

In [2]:
#FETCHES COLUMNS FROM DATASET

#Imports
import pandas as pd
import matplotlib.pyplot as plt

#Get column names
df=pd.read_csv("../data/cleaned/epl-dataset-clean.csv")
print(df.columns)

#Fetches Avg. / Min/Max / Standard Deviation
df.describe()

Index(['Season', 'Pos', 'Team', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GD', 'Pts',
       'Relegation'],
      dtype='str')


,Pos,Pld,W,D,L,GF,GA,GD,Pts
count,520.000000,520.0,500.000000,500.000000,500.000000,500.000000,500.000000,520.000000,520.000000
mean,10.500000,38.0,14.306000,9.388000,14.306000,51.650000,51.650000,0.000000,50.276923
std,5.771834,0.0,6.023032,2.826559,5.650506,16.068725,13.300124,25.792936,19.705545
min,1.000000,38.0,1.000000,2.000000,0.000000,20.000000,15.000000,-69.000000,0.000000
25%,5.750000,38.0,10.000000,7.000000,10.000000,41.000000,43.000000,-17.000000,39.000000
50%,10.500000,38.0,13.000000,9.000000,15.000000,48.000000,51.000000,-4.000000,48.000000
75%,15.250000,38.0,18.000000,11.000000,18.000000,61.250000,60.000000,15.000000,63.000000
max,20.000000,38.0,32.000000,17.000000,30.000000,106.000000,104.000000,79.000000,100.000000


MACHINE LEARNING PRINCPLE:

X = inputs/features
y = target/output

In [3]:
#Defining Target Variable

y = df["Pos"]

print(y.head())

0    1
1    2
2    3
3    4
4    5
Name: Pos, dtype: int64


In [10]:
#Win / Loss / Draw Percentages

df["win_pct"] = df["W"] / 38
df["draw_pct"] = df["D"] / 38
df["loss_pct"] = df["L"] / 38

In [11]:
#Choosing Input Features

features = ["win_pct", "draw_pct", "loss_pct", "GF", "GA", "GD", "Pts"]

x = df[features]

print(x.head())

    win_pct  draw_pct  loss_pct    GF    GA  GD  Pts
0  0.631579  0.210526  0.157895  79.0  31.0  48   80
1  0.526316  0.263158  0.210526  63.0  38.0  25   70
2  0.526316  0.236842  0.236842  71.0  39.0  32   69
3  0.526316  0.210526  0.263158  64.0  43.0  21   68
4  0.526316  0.157895  0.315789  57.0  42.0  15   66


In [17]:
# Remove rows with missing values in X or y

ml_data = df[features + ["Pos"]].dropna()

x = ml_data[features]
y = ml_data["Pos"]

MACHINE LEARNING PRINCPLE:

| Variable  | Meaning          |
| --------- | ---------------- |
| `X_train` | training inputs  |
| `X_test`  | testing inputs   |
| `y_train` | training answers |
| `y_test`  | testing answers  |

1. learn from X_train
2. compare against y_train
3. predict on X_test
4. compare to y_test


In [18]:
# Train/Test Split

from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)

#Verification
print(x_train.shape)
print(x_test.shape)

(400, 7)
(100, 7)


#GOAL

- Model learns W + GD + Pts -> Final League Position

In [19]:
# Train First Machine Learning Model

from sklearn.linear_model import LinearRegression

model = LinearRegression()

model.fit(x_train, y_train)

print("Model training complete.")

Model training complete.


In [29]:
# Make Predictions

y_pred = model.predict(x_test)

# Round predictions to whole league positions
y_pred_rounded = y_pred.round()

# Keep predictions within valid EPL positions: 1 to 20
y_pred_rounded = y_pred_rounded.clip(1, 20)

# Compare actual vs predicted values
results = pd.DataFrame({
    "Actual Position": y_test,
    "Predicted Position": y_pred_rounded
})

print(results.head(10))

     Actual Position  Predicted Position
361                2                 1.0
73                14                13.0
374               15                15.0
155               16                15.0
104                5                 6.0
394               15                14.0
377               18                17.0
124                5                 8.0
68                 9                10.0
450               11                12.0


In [31]:
# Evaluate Model

from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(y_test, y_pred_rounded)

print("Mean Absolute Error:", mae)

Mean Absolute Error: 1.21
